In [0]:
# Definir pastas do projetos em variaveis para facilitar
bronze_path   = '/Volumes/bikestore/default/bikestore/bronze/'
silver_path   = '/Volumes/bikestore/default/bikestore/silver/'
gold_path     = '/Volumes/bikestore/default/bikestore/gold/'
resource_path = '/Volumes/bikestore/default/bikestore/resource/origem/'

In [0]:
#Criando dfs referente pra cada bronze
import pyspark.sql.functions as F

df1 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/brand/")
df2 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/customers/")
df3 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/orders/")
df4 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/orders_item/")
df5 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/products/")
df6 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/stores/")
df7 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/staffs/")
df8 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/categories/")
df9 = spark.read.format("delta").load("/Volumes/bikestore/default/bikestore/bronze/stocks/")


In [0]:
df_order_items = df4\
.select('order_id','item_id','product_id','quantity','list_price','discount')\
.withColumn("total_sale",F.round((F.col("list_price")*F.col("quantity"))*(1-F.col("discount")),2))

In [0]:
df_orders = df3.select("order_id","customer_id","order_status","order_date","required_date","shipped_date",'store_id','staff_id')\
.withColumn('status',F.when(F.col("order_status") == 1, 'Pending')
            .when(F.col("order_status") == 2, 'Processing')
            .when(F.col("order_status") == 3, 'Shipped')
            .when(F.col("order_status") == 4, 'Delivered')
            .otherwise('Unknown'))

In [0]:
df_stores = df6.select("store_id","store_name","city","state")
df_staff = df7.select("staff_id","first_name","active","email")

In [0]:
df_final = df_orders.join(df_order_items,on="order_id",how="left")\
.join(df_stores,on="store_id",how="left")\
.join(df_staff,on="staff_id",how="left")\
.select('order_id', 'customer_id', 'status', 'order_status', 'order_date', 'required_date', 'shipped_date', 'store_name', 'state', 'city', 'first_name', 'active', 'email', 'product_id', 'quantity', 'total_sale', 'list_price', 'discount')


In [0]:
# salvar em Delta na silver 
df_final.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema','true')\
    .save(f'{silver_path}orders')

In [0]:
#criando tabela
df = df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bikestore.logistics.silver_orders")